In [1]:
import tensorflow as tf
import os
import shutil

2025-12-03 02:21:47.458225: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-03 02:21:47.492752: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Import your model class
from sngan.generator_gumbel import GumbelGenerator

# 1. Setup Dummy Flags/Config
class FakeFlags:
    z_dim = 128
    gf_dim = 64
    seq_len = 160
    vocab_size = 21
    # ... add other necessary flags if init requires them ...
    # (Just ensures init doesn't crash)
    attn_pos = 2
    model_type = 'wgan'
    architecture = 'gumbel'
    logdir = '/project/animesh_ray_1465/Zihao/GAN/logs'

In [6]:
def test_deep_save():
    print("--- STARTING PARANOIA CHECK (UPDATED) ---")
    
    # Clean up old test dir
    TEST_DIR = FLAGS.logdir + "./test_checkpoint_verification"
    if os.path.exists(TEST_DIR): shutil.rmtree(TEST_DIR)
    os.makedirs(TEST_DIR)

    # ----------------------------------------
    # PHASE 1: Create Model & Modify a Deep Weight
    # ----------------------------------------
    dummy_shape = [1, 1, 160, 21]
    model_1 = GumbelGenerator(FLAGS, dummy_shape)
    
    # FORCE BUILD
    print("1. Building Model 1...")
    _ = model_1(tf.random.normal([1, 128]), training=False)
    
    # === THE CHANGE IS HERE ===
    # We access the named attribute 'res_block_0' directly
    # instead of the list 'res_blocks[0]'
    target_var = model_1.res_block_0.deconv.w
    # ==========================
    
    # Modify it specifically so we know it's ours (set to all 5.0)
    print("2. Modifying deep variable (res_block_0/deconv/w) to 5.0...")
    target_var.assign(tf.ones_like(target_var) * 5.0)
    
    # Verify modification
    val_check = model_1.res_block_0.deconv.w.numpy()[0,0,0,0]
    print(f"   Model 1 Value: {val_check} (Should be 5.0)")

    # Save
    ckpt_1 = tf.train.Checkpoint(generator=model_1)
    manager_1 = tf.train.CheckpointManager(ckpt_1, TEST_DIR, max_to_keep=1)
    save_path = manager_1.save()
    print(f"3. Saved checkpoint to {save_path}")
    
    # ----------------------------------------
    # PHASE 2: Load into a FRESH Model
    # ----------------------------------------
    print("4. Creating fresh Model 2...")
    model_2 = GumbelGenerator(FLAGS, dummy_shape)
    _ = model_2(tf.random.normal([1, 128]), training=False)
    
    # Check that it is NOT 5.0 yet
    val_random = model_2.res_block_0.deconv.w.numpy()[0,0,0,0]
    print(f"   Model 2 Initial Value: {val_random} (Should NOT be 5.0)")
    
    # Restore
    ckpt_2 = tf.train.Checkpoint(generator=model_2)
    ckpt_2.restore(save_path).assert_consumed() 
    print("5. Restored weights...")
    
    # ----------------------------------------
    # PHASE 3: The Verdict
    # ----------------------------------------
    val_restored = model_2.res_block_0.deconv.w.numpy()[0,0,0,0]
    print(f"   Model 2 Final Value: {val_restored}")
    
    if val_restored == 5.0:
        print("\n✅ SUCCESS: Object-Based Checkpointing successfully saved the inner named variables!")
    else:
        print("\n❌ FAILURE: The inner variables were not saved.")

In [7]:
FLAGS = FakeFlags()

In [8]:
test_deep_save()

--- STARTING PARANOIA CHECK (UPDATED) ---
1. Building Model 1...
2. Modifying deep variable (res_block_0/deconv/w) to 5.0...
   Model 1 Value: 5.0 (Should be 5.0)
3. Saved checkpoint to ./test_checkpoint_verification/ckpt-1
4. Creating fresh Model 2...
   Model 2 Initial Value: -0.01156855933368206 (Should NOT be 5.0)
5. Restored weights...
   Model 2 Final Value: 5.0

✅ SUCCESS: Object-Based Checkpointing successfully saved the inner named variables!
